In [ ]:
%load_ext autoreload
%autoreload 3 -p

In [ ]:
import xarray as xr
import colorcet as cc
import holoviews as hv
import numpy as np
import pandas as pd
from holoviews import opts
from joblib import delayed
from omegaconf import OmegaConf
from tqdm_joblib import ParallelPbar

import gnss_tid.fft
import gnss_tid.image
import gnss_tid.pointdata
import gnss_tid.spectral

hv.extension("bokeh")
opts.defaults(
    opts.Image(
        data_aspect=1,
        frame_height=400,
        frame_width=600,
        cmap=cc.cm.coolwarm,
        colorbar=True,
        show_grid=True,
        tools=["hover"],
    ),
    hv.opts.Scatter(
        data_aspect=1,
        frame_height=400,
        frame_width=600,
        cmap=cc.cm.coolwarm,
        colorbar=True,
        show_grid=True,
        tools=["hover"],
    ),
)

In [ ]:
# cfg = OmegaConf.load("apps/conf/event/event_20170820.yaml")
# cfg = OmegaConf.load("apps/conf/event/event_20160508.yaml")
cfg = OmegaConf.load("apps/conf/event/event_20210101.yaml")
# cfg = OmegaConf.load("apps/conf/event/20150325_full.yaml")
cfg

In [ ]:
lat_limits = cfg.latitude_limits
lon_limits = cfg.longitude_limits
# lon_limits = [-100, -70]
time_limits = cfg.time_limits
# time_limits = ["20210101_160000", "20210101_220000"]

obs, rx = gnss_tid.pointdata.load_observations(
    files=cfg.files,
    lat_limits=lat_limits,
    lon_limits=lon_limits,
    time_limits=time_limits,
    el_min=30,
)

In [ ]:
proj = gnss_tid.pointdata.Local2D.from_geodetic(
    np.mean(lat_limits), np.mean(lon_limits), 300.0
)
windows = gnss_tid.pointdata.make_time_windows(obs, 10, 30)
maker = gnss_tid.image.ScipyRbfImageMaker(
    hres=20,  # grid resolution (km / pixel)
    hp_freq=0.01,  # Butterworth high-pass cutoff (cycles / pixel)
    neighbor_radius=60.0,  # density-mask radius (km)
    neighbors=50,  # RBF neighborhood size
    kernel="multiquadric",
    smoothing=0.3,
)
maker.initialize_from_bounds(proj, lat_limits, lon_limits)

In [ ]:
PLOT_TYPE = "time"
N_JOBS = 8
HEIGHT = 350.0
heights = np.arange(150, 600, 10)
twin = windows[16]

# Patch density gate, matching make_animation/event.gif
# (apps/conf/focus/timesmooth.yaml).  compute_patch_spectra cuts pixels with
# density <= DENSITY_THRESH from a local copy of the image before the FFT
# (zero-filled; the stored image stays unmasked) and records each patch's
# coverage; extract_patch_peaks then gates F/Fx/Fy to NaN wherever coverage
# falls below MIN_PATCH_COVERAGE.
DENSITY_THRESH = 2
MIN_PATCH_COVERAGE = 0.25
BLOCK_SIZE = 32
BLOCK_STEP = 16
KAISER_BETA = 2.0
fft_window = gnss_tid.fft.make_kaiser_2d(BLOCK_SIZE, KAISER_BETA).reshape(
    1, 1, BLOCK_SIZE, BLOCK_SIZE
)


def focus_slice(t, h):
    data = gnss_tid.pointdata.get_data(obs, rx, t, h, lat_limits, lon_limits, proj=proj)
    img = gnss_tid.image.generate_image(data, maker, cfg.tec_name)
    spectra = gnss_tid.spectral.compute_patch_spectra(
        img, BLOCK_SIZE, BLOCK_STEP, fft_window, density_thresh=DENSITY_THRESH
    )
    peaks = gnss_tid.spectral.extract_patch_peaks(
        spectra, min_coverage=MIN_PATCH_COVERAGE
    )
    return img.assign(F=peaks.F, Fx=peaks.Fx, Fy=peaks.Fy)


def ft(t):
    return focus_slice(t, HEIGHT)


def fh(h):
    return focus_slice(twin, h)


if PLOT_TYPE == "time":
    slices = ParallelPbar("making plots")(n_jobs=N_JOBS)(delayed(ft)(t) for t in windows)
    slider = [t.start_time for t in windows]
    title = f"height = {HEIGHT}km"
else:
    slices = ParallelPbar("making plots")(n_jobs=N_JOBS)(delayed(fh)(h) for h in heights)
    slider = list(heights)
    title = np.datetime_as_string(twin.start_time, unit="m")

slices = xr.concat(slices, dim=PLOT_TYPE)

In [ ]:
CLIM = 0.5
F_MAX = slices.F.max().item()


def stacked_plot(ds):
    power = hv.Image(ds.F, kdims=["px", "py"]).opts(clim=(0, F_MAX), cmap=cc.cm.gouldian)

    masked = ds.image.where(ds.density > DENSITY_THRESH)
    image_el = hv.Image(masked, kdims=["x", "y"]).opts(
        title=f"focused image | density > {DENSITY_THRESH:g}"
    )
    return (power + image_el).cols(1)


plots = {slider[i]: stacked_plot(slices.isel(time=i)) for i in range(len(slider))}
hv.output(widget_location="bottom")
hv.HoloMap(plots, kdims=[PLOT_TYPE]).collate()